<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/01_Random_Forest_CHL_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ======================================================================
# STEP 31.9P-12 — RANDOM FOREST BASELINE MODEL
# ======================================================================

import sys
import platform

print("=" * 70)
print("STEP 31.9P-12 — RANDOM FOREST BASELINE MODEL")
print("=" * 70)

print("\nPython version:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

print("\nRandom Forest training device:")
print("CPU")

STEP 31.9P-12 — RANDOM FOREST BASELINE MODEL

Python version:
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

Platform:
Linux-6.6.122+-x86_64-with-glibc2.39

Random Forest training device:
CPU


In [2]:
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [3]:
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESS_DIR = BASE_DIR / "05_model_preprocessing"

MODEL_DIR = (
    BASE_DIR /
    "06_models" /
    "01_Random_Forest"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project directory:")
print(BASE_DIR)

print("\nPreprocessing directory:")
print(PREPROCESS_DIR)

print("\nRandom Forest model directory:")
print(MODEL_DIR)

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Preprocessing directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing

Random Forest model directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest


In [4]:
import pandas as pd
import numpy as np

In [5]:
TRAIN_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_train.csv"
)

VALIDATION_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_validation.csv"
)

TEST_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_test.csv"
)

train_df = pd.read_csv(TRAIN_FILE)
validation_df = pd.read_csv(VALIDATION_FILE)
test_df = pd.read_csv(TEST_FILE)

print("=" * 70)
print("MODEL-READY DATASETS LOADED")
print("=" * 70)

print("\nTraining:")
print(train_df.shape)

print("\nValidation:")
print(validation_df.shape)

print("\nTest:")
print(test_df.shape)

MODEL-READY DATASETS LOADED

Training:
(736950, 75)

Validation:
(157896, 75)

Test:
(157998, 75)


In [6]:
FEATURE_FILE = (
    PREPROCESS_DIR /
    "final_model_feature_columns.csv"
)

feature_df = pd.read_csv(FEATURE_FILE)

FEATURES = (
    feature_df["feature"]
    .dropna()
    .astype(str)
    .tolist()
)

FEATURES = list(dict.fromkeys(FEATURES))

print("=" * 70)
print("FINAL MODEL FEATURES")
print("=" * 70)

print("Number of features:", len(FEATURES))

for i, feature in enumerate(FEATURES, start=1):
    print(f"{i:02d}. {feature}")

FINAL MODEL FEATURES
Number of features: 69
01. sst
02. sss
03. nitrate
04. phosphate
05. silicate
06. year
07. month
08. day
09. day_of_year
10. day_of_week
11. week_of_year
12. day_of_year_sin
13. day_of_year_cos
14. month_sin
15. month_cos
16. day_of_week_sin
17. day_of_week_cos
18. chl_lag_1
19. chl_lag_2
20. chl_lag_3
21. chl_lag_5
22. chl_lag_7
23. chl_lag_14
24. chl_lag_30
25. sst_lag_1
26. sst_lag_3
27. sst_lag_7
28. sst_lag_14
29. sss_lag_1
30. sss_lag_3
31. sss_lag_7
32. sss_lag_14
33. nitrate_lag_1
34. nitrate_lag_3
35. nitrate_lag_7
36. nitrate_lag_14
37. phosphate_lag_1
38. phosphate_lag_3
39. phosphate_lag_7
40. phosphate_lag_14
41. silicate_lag_1
42. silicate_lag_3
43. silicate_lag_7
44. silicate_lag_14
45. chl_rolling_mean_3
46. chl_rolling_mean_7
47. chl_rolling_mean_14
48. chl_rolling_std_7
49. chl_rolling_std_14
50. sst_rolling_mean_3
51. sst_rolling_mean_7
52. sst_rolling_mean_14
53. sss_rolling_mean_3
54. sss_rolling_mean_7
55. sss_rolling_mean_14
56. nitrate_rolli

In [7]:
TARGET = "chlorophyll_a_log1p"
ORIGINAL_TARGET = "chlorophyll_a"

print("=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("Model target:", TARGET)
print("Original CHL target:", ORIGINAL_TARGET)

TARGET CONFIGURATION
Model target: chlorophyll_a_log1p
Original CHL target: chlorophyll_a


In [8]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_validation = validation_df[FEATURES]
y_validation = validation_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("=" * 70)
print("MODEL INPUT SHAPES")
print("=" * 70)

print("\nX_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

MODEL INPUT SHAPES

X_train: (736950, 69)
y_train: (736950,)

X_validation: (157896, 69)
y_validation: (157896,)

X_test: (157998, 69)
y_test: (157998,)


In [9]:
print("=" * 70)
print("PRE-TRAINING SAFETY CHECKS")
print("=" * 70)

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Validation:", X_validation.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

print("\nTarget missing values:")
print("Train:", y_train.isna().sum())
print("Validation:", y_validation.isna().sum())
print("Test:", y_test.isna().sum())

assert X_train.isna().sum().sum() == 0
assert X_validation.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

assert y_train.isna().sum() == 0
assert y_validation.isna().sum() == 0
assert y_test.isna().sum() == 0

print("\n✓ No missing predictor values.")
print("✓ No missing target values.")
print("✓ Ready for Random Forest training.")

PRE-TRAINING SAFETY CHECKS

Missing values:
Train: 0
Validation: 0
Test: 0

Target missing values:
Train: 0
Validation: 0
Test: 0

✓ No missing predictor values.
✓ No missing target values.
✓ Ready for Random Forest training.


In [10]:
# ======================================================================
# STEP 31.9P-12A — RANDOM FOREST TRAINING
# ======================================================================

import time
import joblib
from sklearn.ensemble import RandomForestRegressor

print("=" * 70)
print("STEP 31.9P-12A — RANDOM FOREST TRAINING")
print("=" * 70)

# ----------------------------------------------------------------------
# RANDOM FOREST CONFIGURATION
# ----------------------------------------------------------------------

RF_CONFIG = {
    "n_estimators": 300,
    "max_depth": 20,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "random_state": 42,
    "n_jobs": -1
}

print("\n" + "=" * 70)
print("RANDOM FOREST CONFIGURATION")
print("=" * 70)

for parameter, value in RF_CONFIG.items():
    print(f"{parameter:20s}: {value}")

# ----------------------------------------------------------------------
# CREATE MODEL
# ----------------------------------------------------------------------

rf_model = RandomForestRegressor(
    **RF_CONFIG
)

print("\n" + "=" * 70)
print("MODEL CREATED")
print("=" * 70)

print("Model type:")
print(type(rf_model).__name__)

# ----------------------------------------------------------------------
# TRAIN MODEL
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING RANDOM FOREST")
print("=" * 70)

print("Training rows :", X_train.shape[0])
print("Training features :", X_train.shape[1])
print("Training target :", TARGET)

print("\nCPU parallel processing:")
print("n_jobs =", RF_CONFIG["n_jobs"])

print("\nTraining started...")

training_start = time.time()

rf_model.fit(
    X_train,
    y_train
)

training_end = time.time()

training_time_seconds = training_end - training_start
training_time_minutes = training_time_seconds / 60

print("\n✓ Random Forest training completed.")

print(f"\nTraining time:")
print(f"{training_time_seconds:.2f} seconds")
print(f"{training_time_minutes:.2f} minutes")

# ----------------------------------------------------------------------
# VALIDATION PREDICTIONS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATING VALIDATION PREDICTIONS")
print("=" * 70)

validation_prediction_start = time.time()

y_validation_pred_log = rf_model.predict(
    X_validation
)

validation_prediction_end = time.time()

prediction_time_seconds = (
    validation_prediction_end -
    validation_prediction_start
)

print("✓ Validation predictions generated.")

print(
    f"Prediction time: "
    f"{prediction_time_seconds:.2f} seconds"
)

print("\nPrediction shape:")
print(y_validation_pred_log.shape)

# ----------------------------------------------------------------------
# BACK-TRANSFORM TO ORIGINAL CHL SCALE
# ----------------------------------------------------------------------

y_validation_pred_chl = np.expm1(
    y_validation_pred_log
)

y_validation_actual_chl = validation_df[
    ORIGINAL_TARGET
].to_numpy()

# Prevent tiny numerical negative values
y_validation_pred_chl = np.maximum(
    y_validation_pred_chl,
    0
)

print("\n" + "=" * 70)
print("VALIDATION PREDICTION RANGE")
print("=" * 70)

print(
    "Predicted log1p CHL:"
)

print(
    f"Minimum: {y_validation_pred_log.min():.6f}"
)

print(
    f"Maximum: {y_validation_pred_log.max():.6f}"
)

print(
    "\nPredicted original CHL:"
)

print(
    f"Minimum: {y_validation_pred_chl.min():.6f} mg/m³"
)

print(
    f"Maximum: {y_validation_pred_chl.max():.6f} mg/m³"
)

print(
    "\nActual validation CHL:"
)

print(
    f"Minimum: {y_validation_actual_chl.min():.6f} mg/m³"
)

print(
    f"Maximum: {y_validation_actual_chl.max():.6f} mg/m³"
)

# ----------------------------------------------------------------------
# SAVE MODEL
# ----------------------------------------------------------------------

MODEL_FILE = (
    MODEL_DIR /
    "random_forest_chlorophyll_model.joblib"
)

joblib.dump(
    rf_model,
    MODEL_FILE
)

print("\n" + "=" * 70)
print("MODEL SAVED")
print("=" * 70)

print(MODEL_FILE)

# ----------------------------------------------------------------------
# SAVE MODEL CONFIGURATION
# ----------------------------------------------------------------------

config_df = pd.DataFrame([
    {
        "model": "Random Forest",
        "target": TARGET,
        "number_of_features": len(FEATURES),
        "training_rows": len(X_train),
        "validation_rows": len(X_validation),
        "n_estimators": RF_CONFIG["n_estimators"],
        "max_depth": RF_CONFIG["max_depth"],
        "min_samples_leaf": RF_CONFIG["min_samples_leaf"],
        "max_features": RF_CONFIG["max_features"],
        "random_state": RF_CONFIG["random_state"],
        "n_jobs": RF_CONFIG["n_jobs"],
        "training_time_seconds": training_time_seconds,
        "training_time_minutes": training_time_minutes
    }
])

CONFIG_FILE = (
    MODEL_DIR /
    "random_forest_training_configuration.csv"
)

config_df.to_csv(
    CONFIG_FILE,
    index=False
)

print("\nConfiguration saved:")
print(CONFIG_FILE)

# ----------------------------------------------------------------------
# SAVE VALIDATION PREDICTIONS
# ----------------------------------------------------------------------

validation_predictions = pd.DataFrame({
    "date": validation_df["date"].values,
    "latitude": validation_df["latitude"].values,
    "longitude": validation_df["longitude"].values,
    "actual_chlorophyll_a": y_validation_actual_chl,
    "actual_chlorophyll_a_log1p": y_validation[TARGET].values,
    "predicted_chlorophyll_a_log1p": y_validation_pred_log,
    "predicted_chlorophyll_a": y_validation_pred_chl
})

VALIDATION_PREDICTIONS_FILE = (
    MODEL_DIR /
    "random_forest_validation_predictions.csv"
)

validation_predictions.to_csv(
    VALIDATION_PREDICTIONS_FILE,
    index=False
)

print("\nValidation predictions saved:")
print(VALIDATION_PREDICTIONS_FILE)

# ----------------------------------------------------------------------
# FINAL STATUS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 31.9P-12A COMPLETE")
print("=" * 70)

print("✓ Random Forest trained using training data only.")
print("✓ Validation predictions generated.")
print("✓ Predictions converted back to original CHL scale.")
print("✓ Trained model saved.")
print("✓ Training configuration saved.")
print("✓ Validation predictions saved.")

print("\nIMPORTANT:")
print("✓ Test data has NOT been used.")
print("✓ Test predictions have NOT been generated.")
print("✓ Final test evaluation remains untouched.")

print("\nNext:")
print("STEP 31.9P-12B — RANDOM FOREST VALIDATION EVALUATION")

STEP 31.9P-12A — RANDOM FOREST TRAINING

RANDOM FOREST CONFIGURATION
n_estimators        : 300
max_depth           : 20
min_samples_leaf    : 5
max_features        : sqrt
random_state        : 42
n_jobs              : -1

MODEL CREATED
Model type:
RandomForestRegressor

TRAINING RANDOM FOREST
Training rows : 736950
Training features : 69
Training target : chlorophyll_a_log1p

CPU parallel processing:
n_jobs = -1

Training started...

✓ Random Forest training completed.

Training time:
2555.58 seconds
42.59 minutes

GENERATING VALIDATION PREDICTIONS
✓ Validation predictions generated.
Prediction time: 6.75 seconds

Prediction shape:
(157896,)

VALIDATION PREDICTION RANGE
Predicted log1p CHL:
Minimum: 0.092716
Maximum: 2.209249

Predicted original CHL:
Minimum: 0.097150 mg/m³
Maximum: 8.108876 mg/m³

Actual validation CHL:
Minimum: 0.066738 mg/m³
Maximum: 37.501720 mg/m³

MODEL SAVED
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/random_forest_chlorophyll_mo

KeyError: 'chlorophyll_a_log1p'

In [11]:
# ======================================================================
# STEP 31.9P-12A RECOVERY — SAVE VALIDATION PREDICTIONS
# ======================================================================

print("=" * 70)
print("STEP 31.9P-12A RECOVERY — SAVE VALIDATION PREDICTIONS")
print("=" * 70)

# y_validation is already a Series
y_validation_actual_log = y_validation.to_numpy()

validation_predictions = pd.DataFrame({
    "date": validation_df["date"].values,
    "latitude": validation_df["latitude"].values,
    "longitude": validation_df["longitude"].values,
    "actual_chlorophyll_a": y_validation_actual_chl,
    "actual_chlorophyll_a_log1p": y_validation_actual_log,
    "predicted_chlorophyll_a_log1p": y_validation_pred_log,
    "predicted_chlorophyll_a": y_validation_pred_chl
})

VALIDATION_PREDICTIONS_FILE = (
    MODEL_DIR /
    "random_forest_validation_predictions.csv"
)

validation_predictions.to_csv(
    VALIDATION_PREDICTIONS_FILE,
    index=False
)

print("\n✓ Validation predictions saved successfully.")

print("\nFile:")
print(VALIDATION_PREDICTIONS_FILE)

print("\nShape:")
print(validation_predictions.shape)

print("\nColumns:")
for column in validation_predictions.columns:
    print("✓", column)

print("\nFirst 5 predictions:")
print(validation_predictions.head())

print("\n" + "=" * 70)
print("STEP 31.9P-12A COMPLETE")
print("=" * 70)

print("✓ Random Forest training completed.")
print("✓ Model saved.")
print("✓ Training configuration saved.")
print("✓ Validation predictions saved.")
print("✓ Test data was NOT used.")

STEP 31.9P-12A RECOVERY — SAVE VALIDATION PREDICTIONS

✓ Validation predictions saved successfully.

File:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/random_forest_validation_predictions.csv

Shape:
(157896, 7)

Columns:
✓ date
✓ latitude
✓ longitude
✓ actual_chlorophyll_a
✓ actual_chlorophyll_a_log1p
✓ predicted_chlorophyll_a_log1p
✓ predicted_chlorophyll_a

First 5 predictions:
         date  latitude  longitude  actual_chlorophyll_a  \
0  2017-07-10  5.833334  79.333336              0.316412   
1  2017-07-10  5.833334  79.416664              0.324167   
2  2017-07-10  5.833334  79.500000              0.329531   
3  2017-07-10  5.833334  79.583336              0.343342   
4  2017-07-10  5.833334  79.666664              0.357804   

   actual_chlorophyll_a_log1p  predicted_chlorophyll_a_log1p  \
0                    0.274910                       0.379250   
1                    0.280783                       0.390727   
2                    0.284826 